In [1]:
print("allok")

allok


In [1]:
import os
import getpass
import pandas as pd

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

from deepeval import evaluate
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase

In [3]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")


In [4]:
RAG_MODEL = "gpt-4.1-mini"
EMBEDDING_MODEL ="text-embedding-3-small"
DEEPEVAL_JUDGE_MODEL = "gpt-4.1-mini"

In [5]:
os.environ[
    "DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"
] = "300"

os.environ[
    "DEEPEVAL_PER_TASK_TIMEOUT_SECONDS_OVERRIDE"
] = "600"

os.environ[
    "DEEPEVAL_RETRY_MAX_ATTEMPTS"
] = "1"

In [6]:
documents = [
    Document(
        page_content="Full-time employees receive 24 paid leaves per calendar year.",
        metadata={"doc_id": "leave_policy"},
    ),
    Document(
        page_content="Employees are allowed to work from home for a maximum of 2 days per week.",
        metadata={"doc_id": "remote_policy"},
    ),
    Document(
        page_content="Employees can claim up to ₹3000 per month for internet reimbursement.",
        metadata={"doc_id": "internet_policy"},
    ),
    Document(
        page_content="The standard probation period for new employees is 6 months.",
        metadata={"doc_id": "probation_policy"},
    ),
    Document(
        page_content="Employees receive ₹1000 per month as mobile reimbursement.",
        metadata={"doc_id": "mobile_policy"},
    ),
    Document(
        page_content="Medical insurance coverage begins from the employee's date of joining.",
        metadata={"doc_id": "insurance_policy"},
    ),
]


In [7]:
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [8]:
vector_store = InMemoryVectorStore(embedding=embeddings)

In [9]:
vector_store.add_documents(documents)

['1a963067-e3dc-489f-9034-82db2c0f79d2',
 'c18e93b3-d86e-4ddc-b274-c294566a711d',
 'fffec764-0ff8-4c62-ba42-28c347a245d4',
 '92abd987-396b-4a25-8166-62faa512cdbe',
 '47968daf-893f-4bfe-a914-efab501c77fa',
 '8c3a0a5e-d6ac-4250-a276-25e6c06d50ed']

In [10]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [11]:
llm = ChatOpenAI(
    model=RAG_MODEL,
    temperature=0,
)

In [12]:
def rag_pipeline(query: str) -> dict:
    retrieved_docs = retriever.invoke(query)

    retrieval_context = [
        doc.page_content
        for doc in retrieved_docs
    ]

    retrieved_doc_ids = [
        doc.metadata.get("doc_id")
        for doc in retrieved_docs
    ]

    context = "\n\n".join(retrieval_context)

    prompt = f"""
You are an HR policy assistant.

Answer the user's question ONLY from the supplied context.

Rules:
1. Do not use outside knowledge.
2. Do not invent policy details.
3. If the answer is not present in the context, say:
   "I don't know based on the provided context."
4. Keep the answer concise.

CONTEXT:
{context}

QUESTION:
{query}
"""

    response = llm.invoke(prompt)

    return {
        "answer": response.content,
        "retrieval_context": retrieval_context,
        "retrieved_doc_ids": retrieved_doc_ids,
    }


In [13]:
sample = rag_pipeline("What is the monthly internet reimbursement limit?")

print("ANSWER:")
print(sample["answer"])

print("\nRETRIEVED DOCS:")
print(sample["retrieved_doc_ids"])

print("\nCONTEXT:")
for chunk in sample["retrieval_context"]:
    print("-", chunk)

ANSWER:
The monthly internet reimbursement limit is ₹3000.

RETRIEVED DOCS:
['internet_policy', 'mobile_policy', 'remote_policy']

CONTEXT:
- Employees can claim up to ₹3000 per month for internet reimbursement.
- Employees receive ₹1000 per month as mobile reimbursement.
- Employees are allowed to work from home for a maximum of 2 days per week.


In [14]:
goldens = [
    Golden(
        input="How many paid leaves does a full-time employee receive?",
        expected_output="A full-time employee receives 24 paid leaves per calendar year.",
    ),
    Golden(
        input="How many work-from-home days are allowed per week?",
        expected_output="Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="What is the monthly internet reimbursement limit?",
        expected_output="Employees can claim up to ₹3000 per month for internet reimbursement.",
    ),
    Golden(
        input="What is the probation period for new employees?",
        expected_output="The standard probation period for new employees is 6 months.",
    ),
    Golden(
        input="Can an employee work remotely for 3 days every week?",
        expected_output="No. Employees can work from home for a maximum of 2 days per week.",
    ),
    Golden(
        input="When does employee medical insurance coverage begin?",
        expected_output="Medical insurance coverage begins from the employee's date of joining.",
    ),
]

dataset = EvaluationDataset(goldens=goldens)

print("Goldens:", len(dataset.goldens))


Goldens: 6


In [15]:
rag_runs = []

In [16]:
dataset

EvaluationDataset(test_cases=[], goldens=[Golden(id=None, input='How many paid leaves does a full-time employee receive?', actual_output=None, expected_output='A full-time employee receives 24 paid leaves per calendar year.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(id=None, input='How many work-from-home days are allowed per week?', actual_output=None, expected_output='Employees can work from home for a maximum of 2 days per week.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None), Golden(id=None, input='What is the monthly internet reimbursement limit?', actual_output=None, expected_output='Employees can claim up to ₹3000 per month for 

In [17]:
dataset.goldens

[Golden(id=None, input='How many paid leaves does a full-time employee receive?', actual_output=None, expected_output='A full-time employee receives 24 paid leaves per calendar year.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None),
 Golden(id=None, input='How many work-from-home days are allowed per week?', actual_output=None, expected_output='Employees can work from home for a maximum of 2 days per week.', context=None, retrieval_context=None, additional_metadata=None, comments=None, tools_called=None, expected_tools=None, source_file=None, name=None, custom_column_key_values=None, multimodal=False, images_mapping=None),
 Golden(id=None, input='What is the monthly internet reimbursement limit?', actual_output=None, expected_output='Employees can claim up to ₹3000 per month for internet reimbursement.', context=None,

In [18]:
for i, golden in enumerate(dataset.goldens, start=1):
    print(f"Running RAG {i}/{len(dataset.goldens)}")
    result = rag_pipeline(golden.input)
    rag_runs.append({
        "input": golden.input,
        "expected_output": golden.expected_output,
        "actual_output": result["answer"],
        "retrieval_context": result["retrieval_context"],
        "retrieved_doc_ids": result["retrieved_doc_ids"],
    })


Running RAG 1/6
Running RAG 2/6
Running RAG 3/6
Running RAG 4/6
Running RAG 5/6
Running RAG 6/6


In [19]:
pd.DataFrame([
    {
        "input": run["input"],
        "expected_output": run["expected_output"],
        "actual_output": run["actual_output"],
        "retrieved_doc_ids": run["retrieved_doc_ids"],
    }
    for run in rag_runs
])

,input,expected_output,actual_output,retrieved_doc_ids
0,How many paid leaves does a full-time employee...,A full-time employee receives 24 paid leaves p...,A full-time employee receives 24 paid leaves p...,"[leave_policy, remote_policy, mobile_policy]"
1,How many work-from-home days are allowed per w...,Employees can work from home for a maximum of ...,Employees are allowed to work from home for a ...,"[remote_policy, leave_policy, internet_policy]"
2,What is the monthly internet reimbursement limit?,Employees can claim up to ₹3000 per month for ...,The monthly internet reimbursement limit is ₹3...,"[internet_policy, mobile_policy, remote_policy]"
3,What is the probation period for new employees?,The standard probation period for new employee...,The probation period for new employees is 6 mo...,"[probation_policy, insurance_policy, remote_po..."
4,Can an employee work remotely for 3 days every...,No. Employees can work from home for a maximum...,"No, employees are allowed to work from home fo...","[remote_policy, internet_policy, leave_policy]"
5,When does employee medical insurance coverage ...,Medical insurance coverage begins from the emp...,Employee medical insurance coverage begins fro...,"[insurance_policy, leave_policy, internet_policy]"


In [ ]:
from deepeval.metrics import (
    AnswerRelevancyMetric,
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)

In [21]:
test_cases = [
    LLMTestCase(
        input=run["input"],
        actual_output=run["actual_output"],
        expected_output=run["expected_output"],
        retrieval_context=run["retrieval_context"],
    )
    for run in rag_runs
]

print("Test cases:", len(test_cases))

Test cases: 6


In [22]:
test_cases

[LLMTestCase(input='How many paid leaves does a full-time employee receive?', actual_output='A full-time employee receives 24 paid leaves per calendar year.', expected_output='A full-time employee receives 24 paid leaves per calendar year.', context=None, retrieval_context=['Full-time employees receive 24 paid leaves per calendar year.', 'Employees are allowed to work from home for a maximum of 2 days per week.', 'Employees receive ₹1000 per month as mobile reimbursement.'], metadata=None, tools_called=None, comments=None, expected_tools=None, token_cost=None, completion_time=None, flaky=False, multimodal=False, name=None, tags=None, mcp_servers=None, mcp_tools_called=None, mcp_resources_called=None, mcp_prompts_called=None, custom_column_key_values=None),
 LLMTestCase(input='How many work-from-home days are allowed per week?', actual_output='Employees are allowed to work from home for a maximum of 2 days per week.', expected_output='Employees can work from home for a maximum of 2 days

In [25]:
DEEPEVAL_JUDGE_MODEL

'gpt-4.1-mini'

In [26]:
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

faithfulness = FaithfulnessMetric(
    threshold=0.85,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_relevancy = ContextualRelevancyMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_precision = ContextualPrecisionMetric(
    threshold=0.75,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

contextual_recall = ContextualRecallMetric(
    threshold=0.80,
    model=DEEPEVAL_JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
)

In [27]:
rag_metrics = [
    answer_relevancy,
    faithfulness,
    contextual_relevancy,
    contextual_precision,
    contextual_recall,
]

In [28]:
first_test_case = test_cases[0]

In [29]:
for metric in rag_metrics:
    print(metric)

In [30]:
for metric in rag_metrics:
    print("\n", "=" * 70)
    print(metric.__class__.__name__)


AnswerRelevancyMetric

FaithfulnessMetric

ContextualRelevancyMetric

ContextualPrecisionMetric

ContextualRecallMetric


In [31]:
for metric in rag_metrics:
    print("\n", "=" * 70)
    print(metric.__class__.__name__)
    metric.measure(first_test_case)
    print("Score:", metric.score)
    print("Passed:", metric.is_successful())
    print("Reason:", metric.reason)

c:\Users\Sunny\evaluation\env\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter 
support
  warnings.warn('install "ipywidgets" for Jupyter support')


AnswerRelevancyMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because the response directly and fully addresses the question about the number of paid leaves for a full-time employee without any irrelevant information.

FaithfulnessMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because there are no contradictions; the actual output fully aligns with the retrieval context. Great job maintaining accuracy!

ContextualRelevancyMetric


Score: 0.3333333333333333
Passed: False
Reason: The score is 0.33 because while the relevant statement 'Full-time employees receive 24 paid leaves per calendar year.' directly answers the question, the presence of unrelated statements about work-from-home days and mobile reimbursement dilutes the overall relevancy.

ContextualPrecisionMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because the first node in retrieval contexts clearly provides the exact number of paid leaves, stating 'Full-time employees receive 24 paid leaves per calendar year,' which directly answers the question. The subsequent nodes, ranked lower, discuss unrelated topics like working from home days and mobile reimbursement, appropriately placed below the relevant node.

ContextualRecallMetric


Score: 1.0
Passed: True
Reason: The score is 1.00 because the sentence in the expected output exactly matches the information provided in node 1 of the retrieval context, with no discrepancies or missing details.


In [32]:
from deepeval import evaluate
from deepeval.evaluate import AsyncConfig

In [33]:
evaluation_result = evaluate(
    test_cases=test_cases[:1],
    metrics=rag_metrics,

    async_config=AsyncConfig(
        run_async=False
    ),
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              How many paid leaves does a full-time employee receive?                              │
│  │     Actual Output:      A full-time employee receives 24 paid leaves per calendar year.                      │
│  │     Expected Output:    A full-time employee receives 24 paid leaves per calendar year.                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.80      │ The score is 1.00 because the response directly...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.85      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Contextual Relevancy │ 0.33  │ 0.75      │ The score is 0.33 because while the relevant          │
│              │                      │       │           │ statement 'Full-time employees receive 24 paid        │
│              │                      │       │           │ leaves per calendar year.' directly answers the       │
│              │                      │       │           │ question, the presence of unrelated statements        │
│              │                      │       │           │ about work-from-home days and mobile reimbursement    │
│              │                      │       │           │ dilutes the overall relevancy of the retrieval        │
│              │                      │       │           │ context.                                              │
│        PASS  │ Contextual Precision │ 1.00  │ 0.75      │ The score is 1.00 because the first node in ret...    │
│        PASS  │ Contextual Recall    │ 1.00  │ 0.80      │ The score is 1.00 because the sentence in the e...    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                         ┃ Average Score        ┃ Pass Rate                                   ┃ Total    │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━ │
│  Answer Relevancy               │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Faithfulness                   │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Contextual Relevancy           │ 0.33                 │ 0.00% | passed=0 | failed=1                 │ 1        │
│  Contextual Precision           │ 1.00                 │ 100.00% | passed=1 | failed=0               │ 1        │
│  Contextual Recall              │ 1.00                 │

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

⚠ WARNING: No hyperparameters logged.
» ]8;id=8686411;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 25.48s | token cost: 0.0036908 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [34]:
evaluation_result = evaluate(
    test_cases=test_cases,
    metrics=rag_metrics,

    async_config=AsyncConfig(
        run_async=False
    ),
)

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4.1-mini, strict=False, async_mode=False)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4.1-mini, strict=False, 
async_mode=False)...

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:              How many paid leaves does a full-time employee receive?                              │
│  │     Actual Output:      A full-time employee receives 24 paid leaves per calendar year.                      │
│  │     Expected Output:    A full-time employee receives 24 paid leaves per calendar year.                      │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.80      │ The score is 1.00 because the response directly...    │
│        PASS  │ Faithfulness         │ 1.00  │ 0.85      │ The score is 1.00 because there are no contradi...    │
│        FAIL  │ Contextual Relevancy │ 0.33  │ 0.75      │ The score is 0.33 because while the relevant          │
│              │                      │       │           │ statement 'Full-time employees receive 24 paid        │
│              │                      │       │           │ leaves per calendar year.' directly answers the       │
│              │                      │       │           │ question, the majority of the retrieval context is    │
│              │                      │       │           │ irrelevant, including details about work-from-home    │
│              │                      │       │           │ days and mobile reimbursement, which do not pertain   │
│              │                      │       │           │ to paid leaves.                                       │
│        PASS  │ Contextual Precision │ 1.00  │ 0.75      │ The score is 1.00 because the first node in ret...    │
│        PASS  │ Contextual Recall    │ 1.00  │ 0.80      │ The score is 1.00 because the sentence in the e...    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How many work-from-home days are allowed per week?                                   │
│  │     Actual Output:      Employees are allowed to work from home for a maximum of 2 days per week.            │
│  │     Expected Output:    Employees can work from home for a maximum of 2 days per week.                       │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.80      │

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

⚠ WARNING: No hyperparameters logged.
» ]8;id=8686413;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 131.14s | token cost: 0.021938000000000003 USD)
» Test Results (6 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 6

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

In [ ]:
evaluation_result